# ❤️ دموی ECG: تشخیص ضربان طبیعی و PVC با شبکه عصبی
این نوت‌بوک برای آموزش دانش‌آموزان ساخته شده است. تمام سیگنال‌ها **مصنوعی** هستند و برای تشخیص پزشکی مناسب نیستند.

**ترتیب موج‌ها:** P–QRS–T

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow:', tf.__version__)

## ۱) ساخت یک ضربان طبیعی
هر موج را با یک برآمدگی ریاضی ساده می‌سازیم: P کوچک، QRS باریک و بلند، و T پهن‌تر.

In [ ]:
FS = 256
t = np.linspace(0, 1, FS, endpoint=False)

def gaussian(t, center, width, amp):
    return amp * np.exp(-0.5 * ((t-center)/width)**2)

def normal_beat(t):
    p = gaussian(t, .18, .035, .18)
    q = gaussian(t, .38, .012, -.12)
    r = gaussian(t, .405, .014, 1.15)
    s = gaussian(t, .435, .015, -.27)
    tw = gaussian(t, .69, .07, .34)
    return p + q + r + s + tw

x = normal_beat(t)
plt.figure(figsize=(12,4))
plt.plot(t, x, color='#e94f64', lw=3)
for label, pos in {'P':.18,'Q':.38,'R':.405,'S':.435,'T':.69}.items():
    y = x[np.argmin(abs(t-pos))]
    plt.annotate(label, (pos,y), xytext=(0,18 if y>=0 else -25), textcoords='offset points', ha='center', fontsize=14, weight='bold')
plt.title('One synthetic normal beat: P–QRS–T')
plt.xlabel('Time (seconds)'); plt.ylabel('Amplitude'); plt.grid(alpha=.2); plt.show()

## ۲) ساخت PVC مصنوعی
PVC را به شکل یک ضربان زودتر و با QRS پهن‌تر و متفاوت می‌سازیم. این فقط یک مدل آموزشی ساده‌شده است.

In [ ]:
def pvc_beat(t):
    return (gaussian(t,.34,.025,-.15) + gaussian(t,.43,.07,1.05)
            + gaussian(t,.54,.06,-.70) + gaussian(t,.78,.10,.22))

plt.figure(figsize=(12,4))
plt.plot(t, normal_beat(t), label='Normal', lw=3, color='#2878b5')
plt.plot(t, pvc_beat(t), label='PVC', lw=3, color='#e94f64')
plt.title('Normal vs synthetic PVC'); plt.xlabel('Time'); plt.ylabel('Amplitude')
plt.legend(); plt.grid(alpha=.2); plt.show()

## ۳) تولید دیتاست
برای هر نمونه، جای موج‌ها، اندازه موج و نویز را کمی تغییر می‌دهیم تا همه نمونه‌ها عین هم نباشند.

In [ ]:
def make_sample(label, noise=0.035):
    stretch = np.random.uniform(.94, 1.06)
    shifted_t = np.clip((t - np.random.uniform(-.015,.015))/stretch, 0, 1)
    signal = normal_beat(shifted_t) if label == 0 else pvc_beat(shifted_t)
    signal *= np.random.uniform(.85, 1.15)
    signal += np.random.normal(0, noise, FS)
    signal += .03*np.sin(2*np.pi*np.random.uniform(.2,.6)*t + np.random.rand()*6.28)
    return signal.astype('float32')

N = 1200
X = np.array([make_sample(i%2) for i in range(N)])
y = np.array([i%2 for i in range(N)])
idx = np.random.permutation(N); X, y = X[idx], y[idx]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
X_train = X_train[...,None]; X_test = X_test[...,None]
print('Train:', X_train.shape, ' Test:', X_test.shape)

In [ ]:
fig, ax = plt.subplots(2,4,figsize=(14,6),sharex=True,sharey=True)
for row,label in enumerate([0,1]):
    ids=np.where(y==label)[0][:4]
    for col,i in enumerate(ids):
        ax[row,col].plot(t,X[i],color='#2878b5' if label==0 else '#e94f64')
        ax[row,col].set_title('Normal' if label==0 else 'PVC'); ax[row,col].grid(alpha=.15)
plt.tight_layout(); plt.show()

## ۴) شبکه عصبی یک‌بعدی کوچک
Conv1D مانند یک ذره‌بین روی بخش‌های کوتاه موج حرکت می‌کند و شکل‌های مهم را پیدا می‌کند.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input((FS,1)),
    tf.keras.layers.Conv1D(16, 9, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.Conv1D(32, 7, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling1D(2),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, validation_split=.2, epochs=12, batch_size=32, verbose=1)
plt.figure(figsize=(8,4))
plt.plot(history.history['accuracy'],label='train')
plt.plot(history.history['val_accuracy'],label='validation')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.ylim(.5,1.02); plt.grid(alpha=.2); plt.legend(); plt.show()

## ۵) ارزیابی روی نمونه‌هایی که مدل قبلاً ندیده است

In [ ]:
prob = model.predict(X_test, verbose=0).ravel()
pred = (prob >= .5).astype(int)
print(classification_report(y_test,pred,target_names=['Normal','PVC']))
ConfusionMatrixDisplay(confusion_matrix(y_test,pred),display_labels=['Normal','PVC']).plot(cmap='Blues')
plt.title('Confusion matrix'); plt.show()

## ۶) آزمایش زنده: یک سیگنال تازه
مقدار `kind` را بین `normal` و `pvc` عوض کن. مقدار `noise` را هم زیاد کن و ببین مدل چه زمانی گیج می‌شود.

In [ ]:
kind = 'pvc'       # 'normal' or 'pvc'
noise = 0.05       # try 0.02, 0.10, 0.20
label = 0 if kind == 'normal' else 1
sample = make_sample(label, noise=noise)
p = float(model.predict(sample[None,:,None],verbose=0)[0,0])
answer = 'PVC' if p>=.5 else 'Normal'
plt.figure(figsize=(12,4)); plt.plot(t,sample,color='#e94f64',lw=2)
plt.title(f'Model answer: {answer} | PVC probability = {p:.1%}')
plt.xlabel('Time'); plt.ylabel('Amplitude'); plt.grid(alpha=.2); plt.show()

## چالش‌های کلاسی
1. نویز را بیشتر کنید. دقت چه تغییری می‌کند؟  
2. تعداد نمونه‌های آموزشی را کم کنید. چه اتفاقی می‌افتد؟  
3. شکل PVC را کمی تغییر دهید. آیا مدل هنوز آن را می‌شناسد؟

### نکتهٔ ایمنی
این پروژه یک شبیه‌سازی آموزشی است. تشخیص آریتمی واقعی نیازمند دادهٔ بالینی، لیدهای مناسب ECG، ارزیابی پزشک و اعتبارسنجی دقیق است.

منابع مفهومی: American Heart Association، NHLBI/NIH و MedlinePlus.